In [11]:
# https://pypi.org/project/pqcrypto/
!pip install pqcrypto

!pip install cryptography


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [19]:
import os
from secrets import compare_digest
# from pqcrypto.kem.mceliece348864 import generate_keypair, encrypt, decrypt
# from pqcrypto.kem.mceliece460896 import generate_keypair, encrypt, decrypt
# from pqcrypto.kem.mceliece6688128 import generate_keypair, encrypt, decrypt
# from pqcrypto.kem.mceliece6960119 import generate_keypair, encrypt, decrypt
from pqcrypto.kem.mceliece8192128 import generate_keypair, encrypt, decrypt
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.backends import default_backend

# ----------------------------------------------------------------------
# Symmetric Helper Functions (AES-256-GCM)
# ----------------------------------------------------------------------
def aes_gcm_encrypt(plaintext_message: bytes, symmetric_key: bytes) -> tuple[bytes, bytes, bytes]:
    """Encrypts arbitrary data using AES-256-GCM with the KEM-derived key."""
    # Ensure the key is exactly 32 bytes (256 bits) for AES-256
    # mceliece8192128 naturally outputs a 32-byte shared secret
    assert len(symmetric_key) == 32
    
    # Generate a secure 96-bit random Initialization Vector (Nonce)
    iv = os.urandom(12)
    
    encryptor = Cipher(
        algorithms.AES(symmetric_key),
        modes.GCM(iv),
        backend=default_backend()
    ).encryptor()
    
    ciphertext = encryptor.update(plaintext_message) + encryptor.finalize()
    return ciphertext, iv, encryptor.tag

def aes_gcm_decrypt(ciphertext: bytes, symmetric_key: bytes, iv: bytes, tag: bytes) -> bytes:
    """Decrypts AES-256-GCM ciphertext and validates structural integrity."""
    decryptor = Cipher(
        algorithms.AES(symmetric_key),
        modes.GCM(iv, tag),
        backend=default_backend()
    ).decryptor()
    return decryptor.update(ciphertext) + decryptor.finalize()


# ----------------------------------------------------------------------
# Hybrid Post-Quantum Execution Flow
# ----------------------------------------------------------------------
def main():
    print("====================================================")
    print("     Classic McEliece + AES-GCM-256 Hybrid Demo     ")
    print("====================================================\n")

    # This is the actual data payload of arbitrary size that we want to protect
    secret_payload = b"The Lord sends poverty and wealth; he humbles and he exalts."
    print(f"[+] Target Data Payload: '{secret_payload.decode()}'")

    # 1. Setup Phase (Receiver/Alice)
    print("\n[*] Alice is generating a McEliece key pair...")
    # mceliece8192128 produces a ~1.35 MB Public Key
    public_key, secret_key = generate_keypair()
    print(f"    -> Alice's Public Key size: {len(public_key):,} bytes (~1.35 MB)")

    # 2. Encapsulation & Symmetric Encryption Phase (Sender/Bob)
    print("\n[*] Bob is encapsulating a shared secret and encrypting the payload...")
    
    # Bob uses Alice's public key to generate the random shared master key
    kem_ciphertext, shared_secret_bob = encrypt(public_key)
    
    # Bob immediately feeds that shared secret into AES-GCM-256 to lock the real data
    aes_ciphertext, iv, auth_tag = aes_gcm_encrypt(secret_payload, shared_secret_bob)

    print(f"    -> KEM Ciphertext size (Sent over wire) : {len(kem_ciphertext)} bytes")
    print(f"    -> AES Ciphertext payload size         : {len(aes_ciphertext)} bytes")
    print(f"    -> Derived Shared Key (Hex)            : {shared_secret_bob.hex()}")

    # ------------------------------------------------------------------
    # TRANSMISSION: Bob sends (kem_ciphertext, aes_ciphertext, iv, auth_tag)
    # ------------------------------------------------------------------

    # 3. Decapsulation & Decryption Phase (Receiver/Alice)
    print("\n[*] Alice received the payload. Processing decryption pipeline...")
    
    # Alice uses her private key to extract the identical shared secret matrix
    shared_secret_alice = decrypt(secret_key, kem_ciphertext)
    
    # Verify constant-time agreement of the key exchange keys before symmetric execution
    if not compare_digest(shared_secret_bob, shared_secret_alice):
        raise ValueError("[🛑] Critical Error: KEM key derivation mismatch.")
        
    print("    [✅] KEM Shared Secrets Match in constant-time!")

    # Alice uses the recovered key to unlock the actual message
    decrypted_payload = aes_gcm_decrypt(aes_ciphertext, shared_secret_alice, iv, auth_tag)
    print(f"    -> Recovered Data Payload              : '{decrypted_payload.decode()}'")

    # Final assertion sanity check
    assert compare_digest(secret_payload, decrypted_payload)
    print("\n[🎉] SUCCESS: Secure hybrid cryptographic channel validated.")

if __name__ == "__main__":
    main()

     Classic McEliece + AES-GCM-256 Hybrid Demo     

[+] Target Data Payload: 'The Lord sends poverty and wealth; he humbles and he exalts.'

[*] Alice is generating a McEliece key pair...
    -> Alice's Public Key size: 1,357,824 bytes (~1.35 MB)

[*] Bob is encapsulating a shared secret and encrypting the payload...
    -> KEM Ciphertext size (Sent over wire) : 208 bytes
    -> AES Ciphertext payload size         : 60 bytes
    -> Derived Shared Key (Hex)            : a054abd50fe6371c360c0370c2d1f1c16fdc2153b80628d2539ae0bf4fa829c6

[*] Alice received the payload. Processing decryption pipeline...
    [✅] KEM Shared Secrets Match in constant-time!
    -> Recovered Data Payload              : 'The Lord sends poverty and wealth; he humbles and he exalts.'

[🎉] SUCCESS: Secure hybrid cryptographic channel validated.
